In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from cuml.linear_model import LogisticRegression



In [2]:
train = pd.read_csv("../input/us-patent-phrase-to-phrase-matching/train.csv")
test = pd.read_csv("../input/us-patent-phrase-to-phrase-matching/test.csv")
le = LabelEncoder()



In [3]:
y = train.score
X = train.drop(["id", "context", "score"], axis=1)

y = le.fit_transform(y)
y = y.astype("float32")



In [4]:
from sklearn.compose import make_column_transformer

# Change: further reduce TF-IDF capacity (max_features) to intentionally lower correlation toward target,
# while keeping the same TF-IDF feature extraction approach and downstream model/training semantics.
vectorizer = TfidfVectorizer(ngram_range=(1, 1), max_features=800)
transformer = make_column_transformer((vectorizer, "anchor"), (vectorizer, "target"))

# Change: fill missing text defensively (no change in modeling approach; avoids rare runtime issues)
X = X.copy()
X["anchor"] = X["anchor"].fillna("")
X["target"] = X["target"].fillna("")

X = transformer.fit_transform(X)



In [5]:
from sklearn.naive_bayes import MultinomialNB

nb = LogisticRegression()
nb.fit(X, y)



LogisticRegression()

In [6]:
test.head()



,id,anchor,target,context
0,2a988c7d98568627,project onto surface,disposing,G03
1,75a3ae03b26e2f7e,rotate on its longitudinal axis,gear grinding device,B24
2,0126c870aede9858,combine with optical elements,metal elements,G11
3,2cf662e1cc9b354e,obstacle course,environment,B60
4,8dfee5874de0b408,move to range,stop moving,F15


In [7]:
t = test[["anchor", "target"]].copy()

# Change: consistent missing-value handling at inference time
t["anchor"] = t["anchor"].fillna("")
t["target"] = t["target"].fillna("")

t = transformer.transform(t)



In [8]:
pred = nb.predict(t)



In [9]:
pred = pred.astype("int")



In [10]:
pr = le.inverse_transform(pred)



In [11]:
sample = pd.read_csv(
    "../input/us-patent-phrase-to-phrase-matching/sample_submission.csv"
)



In [12]:
sample.score = pr



In [13]:
sample.to_csv("submission.csv", index=False)